# Indonesia Rice Early Warning

Sistem peringatan dini ketahanan beras nasional berbasis machine learning.

Proyek ini memprediksi produksi, neraca, dan harga beras Indonesia 1–6 bulan ke depan,
lalu menerjemahkannya menjadi status **Aman / Waspada / Rawan**, termasuk simulasi
dampak skenario iklim (El Niño / La Niña).

# Library

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [4]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (11, 4)

# Data Understanding

Dataset: `data/processed/padi_bulanan_provinsi.csv`
Sumber: BPS (Survei Kerangka Sampel Area / KSA) - publikasi "Luas Panen dan Produksi Padi di Indonesia" edisi 2019–2025 dan WebAPI BPS.
Cakupan: 2018-01 s.d. 2026-10, bulanan, per provinsi. 3.788 baris, 19 kolom.

In [7]:
df = pd.read_csv(ROOT / "data/processed/padi_bulanan_provinsi.csv", parse_dates=["tanggal"])
df.head(3)

,tanggal,tahun,bulan,provinsi,wilayah_34,sumber,status_luas_panen,status_produksi,luas_panen_ha,produksi_padi_ton_gkg,produksi_beras_ton,standing_crop_ha,vegetatif_awal_ha,vegetatif_akhir_ha,generatif_ha,persiapan_lahan_ha,potensi_gagal_panen_ha,lahan_bera_ha,tanaman_selain_padi_ha
0,2018-01-01,2018,1,Aceh,Aceh,publikasi,tetap,tetap,14541.53,76118.47,43624.66,126640.63,57848.0,47653.58,21139.38,14967.25,455.60,0.0,7600.90
1,2018-01-01,2018,1,Bali,Bali,publikasi,tetap,tetap,8156.83,51260.55,28759.76,31954.42,17845.0,8169.80,5939.28,6064.26,0.00,0.0,9451.45
2,2018-01-01,2018,1,Banten,Banten,publikasi,tetap,tetap,14504.41,74385.46,42146.58,122752.08,54977.0,37723.80,30051.73,16650.71,259.88,0.0,25424.35


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3788 entries, 0 to 3787
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   tanggal                 3788 non-null   datetime64[ns]
 1   tahun                   3788 non-null   int64         
 2   bulan                   3788 non-null   int64         
 3   provinsi                3788 non-null   object        
 4   wilayah_34              3788 non-null   object        
 5   sumber                  3788 non-null   object        
 6   status_luas_panen       3788 non-null   object        
 7   status_produksi         3788 non-null   object        
 8   luas_panen_ha           3788 non-null   float64       
 9   produksi_padi_ton_gkg   3788 non-null   float64       
 10  produksi_beras_ton      3788 non-null   float64       
 11  standing_crop_ha        3408 non-null   float64       
 12  vegetatif_awal_ha       3408 non-null   float64 

## Data Description

### Kolom identitas

| Kolom | Penjelasan |
|---|---|
| `tanggal` | Awal bulan periode data, format `YYYY-MM-01`. Nilainya mewakili **satu bulan penuh**, bukan satu hari |
| `tahun` | Tahun (2018–2026) |
| `bulan` | Bulan, 1 = Januari sampai 12 = Desember |
| `provinsi` | Nama provinsi. 34 provinsi pada 2018–2022, 38 provinsi mulai 2023 setelah pemekaran Papua |
| `wilayah_34` | Provinsi induk sebelum pemekaran. Papua Selatan, Papua Tengah, dan Papua Pegunungan dipetakan ke **Papua**; Papua Barat Daya ke **Papua Barat**. Dipakai kalau ingin membandingkan wilayah secara konsisten sepanjang 2018–2026 |

### Kolom asal dan kualitas data

| Kolom | Penjelasan |
|---|---|
| `sumber` | Asal data. `publikasi` = PDF publikasi tahunan BPS (2018–2025). `webapi` = WebAPI BPS (2026) |
| `status_luas_panen` | Kualitas angka luas panen. `tetap` = angka final. `sementara` = hasil survei, masih bisa direvisi. `potensi` = **perkiraan BPS** dari tanaman yang sedang tumbuh, belum dipanen |
| `status_produksi` | Sama seperti di atas, untuk produksi. Pada 2026: Januari–April `sementara`, Mei–Oktober `potensi` |

Baris berstatus `potensi` adalah **forecast versi BPS**, bukan hasil pengukuran; dipakai sebagai pembanding.

### Kolom hasil panen

| Kolom | Satuan | Penjelasan |
|---|---|---|
| `luas_panen_ha` | hektare | Luas sawah yang dipanen pada bulan tersebut |
| `produksi_padi_ton_gkg` | ton GKG | Produksi padi dalam bentuk **Gabah Kering Giling**. Dihitung BPS sebagai luas panen × produktivitas (hasil survei ubinan). **Target utama model** |
| `produksi_beras_ton` | ton beras | Produksi beras siap konsumsi. Setara 57,6% dari GKG, yaitu rendemen giling 64,02% dikurangi susut, benih, pakan ternak, dan penggunaan non-pangan. Untuk 2026 kolom ini **kami hitung sendiri** dengan faktor 0,5762 karena tidak tersedia di API |

### Kolom fase tanam (hasil pengamatan KSA)

Setiap bulan BPS mengamati ribuan titik sampel dan mencatat kondisi lahannya. Kolom berikut adalah luas lahan pada tiap kondisi. **Hanya tersedia 2018–2025** (dari publikasi PDF); untuk 2026 masih kosong (`NaN`) sampai publikasi edisi 2026 terbit.

| Kolom | Satuan | Penjelasan |
|---|---|---|
| `vegetatif_awal_ha` | hektare | Padi baru ditanam sampai anakan awal. Umumnya dipanen sekitar **3 bulan** kemudian |
| `vegetatif_akhir_ha` | hektare | Padi menjelang berbunga. Umumnya dipanen sekitar **2 bulan** kemudian |
| `generatif_ha` | hektare | Padi sudah berbunga sampai bulir menua. Umumnya dipanen **1 bulan** kemudian |
| `standing_crop_ha` | hektare | Total tanaman padi yang sedang berdiri. **Merupakan penjumlahan tiga kolom di atas** (terverifikasi, selisih ≤ 0,5 ha). Jangan dipakai bersamaan dengan ketiganya sebagai fitur model karena kolinearitas sempurna |
| `persiapan_lahan_ha` | hektare | Lahan sedang diolah atau digenangi, siap ditanami |
| `lahan_bera_ha` | hektare | Lahan sawah dibiarkan kosong (diberakan). Catatan: Januari 2018 bernilai 0 di semua provinsi, kemungkinan **belum didata**, bukan benar-benar nol |
| `tanaman_selain_padi_ha` | hektare | Lahan sawah yang ditanami komoditas lain, misalnya jagung atau sayuran |
| `potensi_gagal_panen_ha` | hektare | Luas tanaman yang berpotensi gagal panen (puso) akibat banjir, kekeringan, atau serangan hama. Di publikasi 2019–2020 istilahnya "Luas Puso"; angkanya identik, jadi digabung jadi satu kolom |